# HumorVibes — validation: do measured signals predict human funniness?

Human-rated dataset (Humicroedit/FunLines SemEval funniness grades) × the Gemma instrument. Correlate measured **laugh_score** and each raw signal (S surprise, R resolution net-of-null, E efficiency) against the human grade. This is the falsification test for the whole theory: if R doesn't track funniness, the resolution claim is wrong.

In [ ]:
import glob, os, sys, json, time, numpy as np
src = glob.glob('/kaggle/input/**/mesh_signals.py', recursive=True)
assert src, 'attach punchline-mesh-src'
sys.path.insert(0, os.path.dirname(src[0]))
os.environ['GEMMA_PROVIDER']='transformers'
gcfg=[p for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma' in p.lower()]
os.environ['GEMMA_MODEL_PATH']=os.path.dirname(gcfg[0])
from mesh_signals import get_provider, compute_signals, split_setup_punchline
prov = get_provider('transformers')
print('instrument:', prov.name)

In [ ]:
# load Humicroedit RAW CSV directly (HF load_dataset now rejects script-based datasets).
# Canonical academic mirror (Nabil Hossain, Humicroedit author) + GitHub fallbacks.
import io, re, zipfile, urllib.request, pandas as pd
rows=[]
URLS=['https://cs.rochester.edu/u/nhossain/humicroedit/semeval-2020-task-7-data.zip',
      'https://github.com/n-CManey/humicroedit/raw/master/data/task-1/train.csv']
def apply_edit(orig, edit):
    return re.sub(r'<[^/>]*/>', edit, str(orig))
df=None
for u in URLS:
    try:
        raw=urllib.request.urlopen(urllib.request.Request(u, headers={'User-Agent':'HumorVibes research'}), timeout=60).read()
        if u.endswith('.zip'):
            z=zipfile.ZipFile(io.BytesIO(raw))
            name=[n for n in z.namelist() if n.endswith('train.csv') and ('task-1' in n or 'subtask-1' in n)]
            if not name: name=[n for n in z.namelist() if n.endswith('train.csv')]
            df=pd.read_csv(z.open(name[0]))
        else:
            df=pd.read_csv(io.BytesIO(raw))
        print('loaded csv from', u, '| cols:', list(df.columns)[:6]); break
    except Exception as e:
        print('miss', u, str(e)[:80])
assert df is not None, 'could not fetch a rated humor CSV'
gcol='meanGrade' if 'meanGrade' in df.columns else [c for c in df.columns if 'grade' in c.lower()][0]
df=df.dropna(subset=[gcol]).sample(min(180,len(df)), random_state=0)
for _,r in df.iterrows():
    rows.append({'text': apply_edit(r['original'], r['edit']), 'grade': float(r[gcol])})
print('rated items:', len(rows), '| grade range', round(min(x["grade"] for x in rows),2),'-',round(max(x["grade"] for x in rows),2))

In [ ]:
# measure the genome of each item; correlate signals vs human grade
S=[];R=[];E=[];L=[];G=[]
t0=time.time()
for i,r in enumerate(rows):
    setup,punch = split_setup_punchline(r['text'])
    try: sig=compute_signals(prov, setup, punch)
    except Exception: continue
    S.append(sig.surprise_mean); R.append(sig.resolution); E.append(sig.efficiency)
    L.append(sig.laugh_score); G.append(r['grade'])
    if (i+1)%40==0: print(f'{i+1}/{len(rows)} ({time.time()-t0:.0f}s)')
S,R,E,L,G=map(np.array,[S,R,E,L,G])
print('measured', len(G), 'items')

In [ ]:
def corr(a,b):
    def rank(v):
        o=np.argsort(np.argsort(v)); return o
    pear=np.corrcoef(a,b)[0,1]
    spear=np.corrcoef(rank(a),rank(b))[0,1]
    return round(float(pear),3), round(float(spear),3)
res={}
for name,arr in [('laugh_score',L),('S_surprise',S),('R_resolution',R),('E_efficiency',E)]:
    p,s=corr(arr,G); res[name]={'pearson':p,'spearman':s}
    print(f'{name:14s} vs human grade:  pearson {p:+.3f}  spearman {s:+.3f}')
json.dump({'n':int(len(G)),'correlations':res}, open('validation_results.json','w'), indent=2)
best=max(res.items(), key=lambda kv: abs(kv[1]['spearman']))
print('\nBEST predictor of human funniness:', best[0], best[1])

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].scatter(L,G,alpha=0.5,s=18); ax[0].set_xlabel('measured laugh_score'); ax[0].set_ylabel('human grade'); ax[0].set_title('laugh_score vs human funniness')
ax[1].scatter(R,G,alpha=0.5,s=18,c='#e4572e'); ax[1].set_xlabel('R (resolution, net of null)'); ax[1].set_ylabel('human grade'); ax[1].set_title('resolution vs human funniness')
plt.tight_layout(); plt.show()

## Reading it
- A positive correlation of **laugh_score** with the human grade is the headline validation.
- If **R** correlates on its own, the resolution mechanism is doing real work (not just S).
- Humicroedit isolates a one-word edit, so this is a clean test of whether measured surprise/resolution tracks the funniness humans actually assign. The number goes straight in the writeup.